# Subscription agreement extraction

Four real, blank subscription agreements from SEC EDGAR, each with its own layout (underlined blanks,
two-column table, bordered table). One parser reads all of them: it knows the usual wordings of each
label ("Date of Birth", "Subscriber Date of Birth"...) but not where any issuer puts them.

Two methods, compared on the same page:
- **Text layer**: the words embedded in a digitally filled PDF. Exact, but scans and photos have none.
- **OCR**: Tesseract on the page image, after flattening shadows and straightening the page.

In [ ]:
from pathlib import Path
import os, sys
if Path(os.getcwd()).name == "notebooks":
    os.chdir("..")  # run from the repo root so the pipeline imports work
sys.path.insert(0, os.getcwd())
import json
import pandas as pd
from config import DATASET as DEMO  # data/demo if built, otherwise samples/
from extract.subscription import read_subscription
from validate.checks import same, us_date

labels = json.loads((DEMO / "labels.json").read_text())

## Accuracy by method and by how the document arrived

In [ ]:
rows = []
for l in labels:
    path = next(DEMO / l["investor_id"] / n for n, kind in l["files"].items() if kind == "subscription")
    raw = read_subscription(path)
    for field, expected in l["subscription"].items():
        if not expected:
            continue
        for method in ("text_layer", "ocr"):
            if method == "text_layer" and l["condition"] != "digital":
                continue
            value = raw[method].get(field)
            if field == "date_of_birth":
                value, expected_iso = us_date(value), us_date(expected)
            else:
                expected_iso = expected
            rows.append({"form": l["form"], "condition": l["condition"], "method": method,
                         "field": field, "correct": same(value, expected_iso)})
acc = pd.DataFrame(rows)
acc.pivot_table(index="method", columns="condition", values="correct", aggfunc="mean").style.format("{:.0%}")

In [ ]:
acc.pivot_table(index="form", columns="condition", values="correct", aggfunc="mean").style.format("{:.0%}")